# **Multigrams: Full Workflow**
The multigram workflow mirrors the unigram workflow, but with two differences. First, instead of _creating_ a whitelist, you filter the multigram corpus _using_ a whitelist containing the top-_N_ unigrams. Second, the multigram workflow adds a **pivoting** step. Pivoting reorganizes the database so that it's easy to train models by year.

You can also use the `db_head`, `db_peek`, and `db_peek_prefix` functions to query the database manually. For instance, you can learn how many times the phrase "presidential address" appeared in 2011 by querying the key "[2011] presidential address."

## **Setup**
### Imports

In [1]:
%load_ext autoreload
%autoreload 2

from ngramprep.ngram_filter import FilterConfig, PipelineConfig
from ngramprep.ngram_acquire import download_and_ingest_to_rocksdb
from ngramprep.ngram_filter.pipeline.orchestrator import build_processed_db
from ngramprep.ngram_pivot.pipeline import build_pivoted_db
from ngramprep.utilities.peek import db_head, db_peek, db_peek_prefix
from ngramprep.utilities.count_items import count_db_items

### Configure
Here we set basic parameters: the corpus to download, the size of the ngrams to download, and the size of the year bins.

In [2]:
db_path_stub = '/scratch/edk202/NLP_corpora/Google_Books/'
archive_path_stub = None
release = '20200217'
language = 'eng'
ngram_size = 5
bin_size = 1

## **Step 1: Download and Ingest**

Specifying `combined_bigrams_download` will convert compound terms into single, hyphenated tokens. This process is case-sensitive, so we specify all common capitalization patterns.

If you're resuming downloads after an interruption, there may be a lag before you see any output. The RocksDB is being repaired.

In [4]:
combined_bigrams_download = {
    "human being", "Human being", "Human Being", "human beings", "Human beings", "Human Beings"
}

download_and_ingest_to_rocksdb(
    ngram_size=ngram_size,
    repo_release_id=release,
    repo_corpus_id=language,
    db_path_stub=db_path_stub,
    archive_path_stub=None,
    ngram_type="tagged",
    random_seed=239,
    overwrite_db=False,
    workers=50,
    write_batch_size=5_000,
    open_type="write:packed24",
    compact_after_ingest=False,
    combined_bigrams=None
)

N-GRAM ACQUISITION PIPELINE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Start Time: 2026-02-07 16:36:27

Download Configuration
════════════════════════════════════════════════════════════════════════════════════════════════════
Ngram repo:           https://books.storage.googleapis.com/?prefix=ngrams/books/20200217/eng/5-
DB path:              /scratch/edk202/NLP_corpora/Google_Books/20200217/eng/5gram_files/5grams.db
File range:           0 to 19422
Total files:          19423
Files to get:         0
Skipping:             19423
Download workers:     50
Batch size:           5,000
Ngram size:           5
Ngram type:           tagged
Overwrite DB:         False
DB Profile:           write:packed24

Download Progress
════════════════════════════════════════════════════════════════════════════════════════════════════


Files Processed:   0%|                                                               | 0/0 [00:00<?]



Processing complete!

Final Summary
════════════════════════════════════════════════════════════════════════════════════════════════════
Fully processed files:       0
Failed files:                0
Total entries written:       0
Write batches flushed:       0
Uncompressed data processed: 0.00 B
Processing throughput:       0.00 MB/sec

End Time: 2026-02-07 17:13:55.484161
Total Runtime: 0:37:28.230749
Time per file: 0:00:00
Files per hour: 0.0


## **Step 2: Filter and Normalize**
`config.py` contains generic defaults for the filtering pipeline. You can override these defaults by passing option dictionaries to the `build_processed_db` function, as seen below. As implemented here, we use the whitelist from the unigram workflow to filter the multigram corpus. If we weren't using a whitelist, we could normalize, filter, and lemmatize each token just as we did for the unigrams.

`always_include_tokens` is applied after case-normalization, so we use all lowercase.

In [3]:
always_include_tokens = {
    'he', 'she',
    'him', 'her',
    'himself', 'herself',
    'man', 'woman'
}

filter_config = FilterConfig(
    bin_size=bin_size,
    whitelist_path=f'{db_path_stub}{release}/{language}/1gram_files/1grams_processed.db/whitelist.txt',
    always_include=always_include_tokens
)

pipeline_config = PipelineConfig(
    # Path construction
    ngram_size=ngram_size,
    repo_release_id=release,
    repo_corpus_id=language,
    db_path_stub=db_path_stub,
    # Pipeline options
    mode="restart",
    num_workers=36,
    num_initial_work_units=600,
    work_unit_claim_order="random",
    cache_partitions=True,
    use_cached_partitions=True,
    progress_every_s=30.0,
    compact_after_ingest=True,
    # No whitelist output for multigrams
    output_whitelist_path=None
)

build_processed_db(
    filter_config=filter_config,
    pipeline_config=pipeline_config
);


N-GRAM FILTER PIPELINE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Start Time: 2026-02-07 23:55:53
Mode:       RESTART

Configuration
════════════════════════════════════════════════════════════════════════════════════════════════════
Source DB:            /scratch/edk202/NLP_corpora/Google_Books/20200217/eng/5gram_files/5grams.db
Target DB:            ...dk202/NLP_corpora/Google_Books/20200217/eng/5gram_files/5grams_processed.db
Temp directory:       ...tch/edk202/NLP_corpora/Google_Books/20200217/eng/5gram_files/processing_tmp

Parallelism
────────────────────────────────────────────────────────────────────────────────────────────────────
Workers:              36
Initial work units:   600

Database Profiles
────────────────────────────────────────────────────────────────────────────────────────────────────
Reader profile:       read:packed24
Writer profile:       write:packed24

Ingestion Configuration
────────────────────────

RuntimeError: 12 worker processes failed with non-zero exit codes

## **Step 3: Pivot to Yearly Indices**
This function rearranges the filtered data such that each record has a year (or year bin) prefix. Ideal for time-series announcement. 

In [4]:
build_pivoted_db(
    mode="restart",
    ngram_size=ngram_size,
    repo_release_id=release,
    repo_corpus_id=language,
    db_path_stub=db_path_stub,
    num_workers=30,
    num_initial_work_units=600,
    cache_partitions=True,
    use_cached_partitions=False,
    compact_after_ingest=True,
    progress_every_s=30.0,
);


PARALLEL N-GRAM DATABASE PIVOT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Start Time: 2026-02-05 10:03:58
Mode:       RESTART

Configuration
════════════════════════════════════════════════════════════════════════════════════════════════════
Source DB:            ...dk202/NLP_corpora/Google_Books/20200217/rus/5gram_files/5grams_processed.db
Target DB:            .../edk202/NLP_corpora/Google_Books/20200217/rus/5gram_files/5grams_pivoted.db
Temp directory:       /scratch/edk202/NLP_corpora/Google_Books/20200217/rus/5gram_files/pivoting_tmp

Parallelism
────────────────────────────────────────────────────────────────────────────────────────────────────
Workers:              30
Initial work units:   600

Database Profiles
────────────────────────────────────────────────────────────────────────────────────────────────────
Reader profile:       read:packed24
Writer profile:       write:packed24
Ingest profile:       write:packed24



SST Files Ingested: 100%|████████████████████████████████████████████████████| 600/600 [00:24<00:00]



Phase 4: Finalizing database...
════════════════════════════════════════════════════════════════════════════════════════════════════

Post-Ingestion Compaction
────────────────────────────────────────────────────────────────────────────────────────────────────
Initial DB size:         554.25 MB
Compaction completed in 0:00:27
Size before:             554.25 MB
Size after:              1.24 GB
Space saved:             -710.55 MB (-128.2%)

┌──────────────────────────────────────────────────────────────────────────────────────────────────┐
│ PROCESSING COMPLETE                                                                              │
├──────────────────────────────────────────────────────────────────────────────────────────────────┤
│ Items: 27,375,867 (estimated)                                                                    │
│ Size: 1.24 GB                                                                                    │
│ Database: /scratch/edk202/NLP_corpora/Google_Book

## **Optional: Inspect Database Files**

### `db_head`: Show first N records

In [7]:
db = f'{db_path_stub}{release}/{language}/{ngram_size}gram_files/{ngram_size}grams_processed.db'

db_head(db, n=5)

First 5 key-value pairs:
────────────────────────────────────────────────────────────────────────────────────────────────────
[ 1] Key:   <UNK> <UNK> <UNK> abaca banana
     Value: Total: 59 occurrences in 59 volumes (1992-2013, 17 bins)

[ 2] Key:   <UNK> <UNK> <UNK> abaca industry
     Value: Total: 267 occurrences in 223 volumes (1901-2019, 65 bins)

[ 3] Key:   <UNK> <UNK> <UNK> abaca plant
     Value: Total: 487 occurrences in 477 volumes (1887-2019, 111 bins)

[ 4] Key:   <UNK> <UNK> <UNK> abaca plantation
     Value: Total: 69 occurrences in 51 volumes (1898-1987, 19 bins)

[ 5] Key:   <UNK> <UNK> <UNK> abaca production
     Value: Total: 630 occurrences in 550 volumes (1949-2003, 39 bins)



### `db_peek`: Show Records starting from a key

In [8]:
db = f'{db_path_stub}{release}/{language}/{ngram_size}gram_files/{ngram_size}grams_processed.db'

db_peek(db, start_key="à <UNK> <UNK> <UNK> <UNK>", n=5)

5 key-value pairs starting from c3a0203c554e4b3e203c554e4b3e203c554e4b3e203c554e4b3e:
────────────────────────────────────────────────────────────────────────────────────────────────────


### `db_peek_prefix`: Show records matching a prefix

In [10]:
db = f'{db_path_stub}{release}/{language}/{ngram_size}gram_files/{ngram_size}grams_processed.db'

db_peek_prefix(db, prefix="b", n=5)

5 key-value pairs with prefix 62:
────────────────────────────────────────────────────────────────────────────────────────────────────
[ 1] Key:   baa <UNK> <UNK> <UNK> also
     Value: Total: 59 occurrences in 57 volumes (1921-1988, 22 bins)

[ 2] Key:   baa <UNK> <UNK> <UNK> authority
     Value: Total: 299 occurrences in 260 volumes (1967-2019, 38 bins)

[ 3] Key:   baa <UNK> <UNK> <UNK> average
     Value: Total: 284 occurrences in 45 volumes (1953-2003, 23 bins)

[ 4] Key:   baa <UNK> <UNK> <UNK> baa
     Value: Total: 127 occurrences in 121 volumes (1926-2017, 40 bins)

[ 5] Key:   baa <UNK> <UNK> <UNK> better
     Value: Total: 147 occurrences in 121 volumes (1966-2016, 35 bins)



## **Optional: Count Database Items**

In [ ]:
raw_db = f'{db_path_stub}{release}/{language}/{ngram_size}gram_files/{ngram_size}grams.db'

raw_count = count_db_items(raw_db)

DATABASE ITEM COUNTER
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
/scratch/edk202/NLP_corpora/Google_Books/20200217/eng-fiction/5gram_files/5grams.db
Progress interval: every 10,000,000 items

COUNTING
────────────────────────────────────────────────────────────────────────────────────────────────────
[     10,000,000] | elapsed     25.2s | rate 397,507 items/sec
[     20,000,000] | elapsed     47.3s | rate 422,881 items/sec
[     30,000,000] | elapsed     74.4s | rate 402,994 items/sec
[     40,000,000] | elapsed     97.4s | rate 410,517 items/sec
[     50,000,000] | elapsed    118.2s | rate 422,954 items/sec
[     60,000,000] | elapsed    137.2s | rate 437,405 items/sec
[     70,000,000] | elapsed    158.3s | rate 442,239 items/sec
[     80,000,000] | elapsed    185.7s | rate 430,781 items/sec
[     90,000,000] | elapsed    205.5s | rate 437,971 items/sec
[    100,000,000] | elapsed    224.9s | rate 444,696 items/sec
[    110,0

In [3]:
filtered_db = f'{db_path_stub}{release}/{language}/{ngram_size}gram_files/{ngram_size}grams_processed.db'

filtered_count = count_db_items(filtered_db)


DATABASE ITEM COUNTER
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
/scratch/edk202/NLP_corpora/Google_Books/20200217/eng/5gram_files/5grams_processed.db
Progress interval: every 10,000,000 items

COUNTING
────────────────────────────────────────────────────────────────────────────────────────────────────
[     10,000,000] | elapsed     10.1s | rate 991,431 items/sec
[     20,000,000] | elapsed     19.6s | rate 1,020,218 items/sec
[     30,000,000] | elapsed     27.9s | rate 1,073,522 items/sec
[     40,000,000] | elapsed     40.5s | rate 988,649 items/sec
[     50,000,000] | elapsed     49.3s | rate 1,015,175 items/sec
[     60,000,000] | elapsed     57.5s | rate 1,043,303 items/sec
[     70,000,000] | elapsed     66.9s | rate 1,046,662 items/sec
[     80,000,000] | elapsed     75.2s | rate 1,063,220 items/sec
[     90,000,000] | elapsed     84.6s | rate 1,063,860 items/sec
[    100,000,000] | elapsed     93.2s | rate 1,072,987 it

In [4]:
pivoted_db = f'{db_path_stub}{release}/{language}/{ngram_size}gram_files/{ngram_size}grams_pivoted.db'

pivoted_count = count_db_items(pivoted_db, progress_interval=50_000_000)

DATABASE ITEM COUNTER
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
/scratch/edk202/NLP_corpora/Google_Books/20200217/eng/5gram_files/5grams_pivoted.db
Progress interval: every 50,000,000 items

COUNTING
────────────────────────────────────────────────────────────────────────────────────────────────────
[     50,000,000] | elapsed     22.2s | rate 2,254,228 items/sec
[    100,000,000] | elapsed     44.4s | rate 2,251,219 items/sec
[    150,000,000] | elapsed     66.9s | rate 2,243,260 items/sec
[    200,000,000] | elapsed     90.0s | rate 2,222,170 items/sec
[    250,000,000] | elapsed    113.8s | rate 2,196,644 items/sec
[    300,000,000] | elapsed    136.2s | rate 2,202,331 items/sec
[    350,000,000] | elapsed    159.0s | rate 2,201,307 items/sec
[    400,000,000] | elapsed    182.4s | rate 2,192,599 items/sec
[    450,000,000] | elapsed    204.4s | rate 2,201,311 items/sec
[    500,000,000] | elapsed    227.7s | rate 2,196,002 

In [ ]:
pivoted_db = f'{db_path_stub}{release}/{language}/{ngram_size}gram_files/{ngram_size}grams_pivoted.db'

pivoted_count_per_bin = count_db_items(pivoted_db, progress_interval=50_000_000, grouping='year_bin')

DATABASE ITEM COUNTER
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
/scratch/edk202/NLP_corpora/Google_Books/20200217/eng/5gram_files/5grams_pivoted.db
Progress interval: every 50,000,000 items
Grouping by: year_bin

COUNTING
────────────────────────────────────────────────────────────────────────────────────────────────────
[     50,000,000] | elapsed     35.0s | rate 1,427,305 items/sec
[    100,000,000] | elapsed     71.2s | rate 1,405,263 items/sec
[    150,000,000] | elapsed    107.3s | rate 1,398,294 items/sec
[    200,000,000] | elapsed    142.6s | rate 1,402,985 items/sec
[    250,000,000] | elapsed    177.9s | rate 1,405,132 items/sec
[    300,000,000] | elapsed    213.0s | rate 1,408,135 items/sec
[    350,000,000] | elapsed    248.9s | rate 1,406,160 items/sec
[    400,000,000] | elapsed    287.8s | rate 1,389,792 items/sec
[    450,000,000] | elapsed    323.7s | rate 1,389,984 items/sec
[    500,000,000] | elapsed    35